In [0]:
%run "/Workspace/Users/dnp50022@gmail.com/Retail-Sales-Data-Pipeline/databricks/notebooks/silver/service principle"

In [0]:

# COMMAND ----------

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.functions import (col, trim, to_date, row_number, current_timestamp, lit,concat_ws,coalesce)

# COMMAND ----------


storage_account = "salesstorageproject"
container_name = "sales"
table_name = "orderdetails"

bronze_path = f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/bronze data/{table_name}/*"


In [0]:

# Read Transaction CSV from Bronze
df = spark.read.format("csv").option("header", "true").load(bronze_path)  
display(df)

In [0]:
df.columns

In [0]:
# CAST DATATYPES
df = (
    df.withColumn("OrderDetailID", col("OrderDetailID").cast("string"))
      .withColumn("OrderID", col("OrderID").cast("string"))
      .withColumn("ProductID", col("ProductID").cast("string"))
      .withColumn("Quantity", col("Quantity").cast("int"))
      .withColumn("TotalAmount", col("TotalAmount").cast("double"))
      .withColumn("ModifiedDate", to_timestamp(col("ModifiedDate")))
)

In [0]:
# PK FILTER (OrderDetailID must be valid)
df = df.filter(
    col("OrderDetailID").isNotNull() &
    (trim(col("OrderDetailID")) != "") &
    (col("OrderDetailID") != "0")
)

In [0]:
# BUSINESS RULES
df = df.filter(
    (col("Quantity") > 0) &
    (col("TotalAmount") >= 0)
)

In [0]:
# DEDUPLICATION (Keep Latest Record)
w = Window.partitionBy("OrderDetailID").orderBy(
    col("ModifiedDate").desc()
)

df = (
    df.withColumn("row_num", row_number().over(w))
      .filter(col("row_num") == 1)
      .drop("row_num")
)

In [0]:
# ADD DERIVED COLUMN (VERY IMPORTANT FOR GOLD 🚀)
df = df.withColumn(
    "UnitPrice",
    col("TotalAmount") / col("Quantity")
)


In [0]:
# ADD INGESTION TIMESTAMP
df = df.withColumn("ingestion_timestamp", current_timestamp())


In [0]:
# WRITE TO UNITY CATALOG
silver_table = "sales.silver.orderdetails"

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(silver_table)

print("Silver table created:", silver_table)

In [0]:
%sql
select * from sales.silver.orderdetails